# Laboratório: Construção de Matriz de Incidência Binária
**Disciplina:** Processamento de Linguagem Natural (PLN) e Sistemas de Recuperação de Informação (RI)

**Visão Geral:** Este laboratório prático tem como objetivo demonstrar, passo a passo, a construção de um Sistema de Recuperação de Informação Booleano. Vamos ler uma coleção de documentos de texto, processar e normalizar o conteúdo, extrair o vocabulário, construir uma matriz de incidência termo-documento e executar consultas booleanas (AND, OR, NOT).

## Etapa 1: Leitura da Coleção de Documentos
Vamos iniciar lendo os relatórios jurídicos disponíveis no diretório. Cada arquivo `.txt` representará um documento na nossa coleção.

In [1]:
import os
import glob

# Diretório onde os arquivos estão localizados (ajuste conforme necessário)
diretorio = '.'

# Busca por todos os arquivos .txt
arquivos = glob.glob(os.path.join(diretorio, '*.txt'))

colecao = {}
for caminho_arquivo in arquivos:
    nome_arquivo = os.path.basename(caminho_arquivo)
    with open(caminho_arquivo, 'r', encoding='utf-8') as f:
        conteudo = f.read()
        colecao[nome_arquivo] = conteudo

print(f"Total de documentos lidos: {len(colecao)}")

Total de documentos lidos: 10


## Etapa 2: Tokenização, Normalização e Remoção de Stop Words
Nesta etapa, vamos processar o texto de cada documento:
- Converter tudo para letras minúsculas.
- Remover pontuação e acentos (usando `unicodedata`).
- Tokenizar (separar as palavras).
- Remover palavras muito comuns (stop words) que não ajudam na busca.

In [2]:
import re
import unicodedata

# Lista estruturada de stopwords em português
stopwords = set([
    'a', 'o', 'as', 'os', 'de', 'do', 'da', 'dos', 'das', 'em', 'no', 'na', 
    'nos', 'nas', 'por', 'para', 'com', 'um', 'uma', 'uns', 'umas', 'e', 'ou', 
    'mas', 'que', 'se', 'como', 'mais', 'é', 'são', 'foi', 'foram', 'ser', 
    'não', 'sim', 'ao', 'aos', 'à', 'às', 'pelo', 'pela', 'pelos', 'pelas'
])

def normalizar_texto(texto):
    # Converte para minúsculas
    texto = texto.lower()
    # Remove acentos
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    # Remove pontuação e números (mantém apenas letras)
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    return texto

documentos_processados = {}

for doc_id, texto in colecao.items():
    texto_normalizado = normalizar_texto(texto)
    # Tokenização por espaço
    tokens = texto_normalizado.split()
    # Remoção de stopwords e palavras com apenas 1 letra
    tokens_filtrados = [word for word in tokens if word not in stopwords and len(word) > 1]
    documentos_processados[doc_id] = tokens_filtrados

# Exibindo os tokens do primeiro documento como exemplo
primeiro_doc = list(documentos_processados.keys())[0] if documentos_processados else None
if primeiro_doc:
    print(f"Tokens de {primeiro_doc}: {documentos_processados[primeiro_doc][:10]}...")

Tokens de relatorio_08.txt: ['parecer', 'sobre', 'violacao', 'sigilo', 'aplicacao', 'multa', 'rescisoria']...


## Etapa 3: Extração do Vocabulário Único (V)
O vocabulário é o conjunto de todas as palavras únicas que aparecem em nossa coleção. Ele definirá as linhas da nossa Matriz de Incidência.

In [3]:
vocabulario = set()

for tokens in documentos_processados.values():
    vocabulario.update(tokens)

# Convertendo para lista ordenada para facilitar a visualização
vocabulario = sorted(list(vocabulario))

print(f"Tamanho do vocabulário (|V|): {len(vocabulario)} termos únicos.")
print(f"Amostra do dicionário: {vocabulario[:20]}...")

Tamanho do vocabulário (|V|): 39 termos únicos.
Amostra do dicionário: ['antecipada', 'aplicacao', 'auditoria', 'causa', 'clausula', 'cobranca', 'confidencialidade', 'contrato', 'contratual', 'corporativo', 'fatura', 'inadimplencia', 'intelectual', 'justa', 'licenca', 'manual', 'multa', 'notificacao', 'nuvem', 'parecer']...


## Etapa 4: Construção da Matriz de Incidência Binária
Vamos utilizar a biblioteca `pandas` para construir a matriz. As linhas serão os termos do vocabulário e as colunas serão os documentos.
Se um termo aparece em um documento, a célula receberá o valor `1`. Caso contrário, receberá `0`.

In [4]:
import pandas as pd
import numpy as np
from IPython.display import display

# Inicializando uma matriz de zeros
matriz = np.zeros((len(vocabulario), len(documentos_processados)), dtype=int)

# Criando o DataFrame
documentos_nomes = sorted(list(documentos_processados.keys()))
df_matriz = pd.DataFrame(matriz, index=vocabulario, columns=documentos_nomes)

# Preenchendo a matriz
for doc_id, tokens in documentos_processados.items():
    for token in tokens:
        df_matriz.at[token, doc_id] = 1

print("Trecho da Matriz de Incidência Binária:")
display(df_matriz.head(10))

Trecho da Matriz de Incidência Binária:


,relatorio_01.txt,relatorio_02.txt,relatorio_03.txt,relatorio_04.txt,relatorio_05.txt,relatorio_06.txt,relatorio_07.txt,relatorio_08.txt,relatorio_09.txt,relatorio_10.txt
antecipada,0,0,1,0,0,0,0,0,0,0
aplicacao,0,0,0,0,0,0,0,1,0,0
auditoria,0,0,0,0,0,0,0,0,0,1
causa,0,0,0,0,0,1,0,0,0,0
clausula,0,0,1,0,1,0,0,0,0,0
cobranca,0,0,0,1,0,0,0,0,0,0
confidencialidade,0,0,0,0,0,0,1,0,0,0
contrato,0,0,0,0,1,1,0,0,0,1
contratual,1,0,1,0,0,0,0,0,0,0
corporativo,0,1,0,0,0,0,0,0,0,0


## Etapa 5: Exportação da Matriz
Agora, vamos exportar a matriz construída para um arquivo CSV chamado `matriz_incidencia.csv`, o que permitirá sua reutilização futura.

In [5]:
caminho_csv = 'matriz_incidencia.csv'
df_matriz.to_csv(caminho_csv)
print(f"Matriz de incidência exportada com sucesso para: {caminho_csv}")

Matriz de incidência exportada com sucesso para: matriz_incidencia.csv


## Etapa 6: Execução de Consultas Booleanas
Utilizaremos a matriz de incidência para resolver as consultas. Os operadores booleanos serão aplicados usando as operações Bitwise no Pandas (`&` para AND, `|` para OR, `~` para NOT).

In [6]:
def buscar_documentos(resultado_booleano):
    # Retorna os nomes das colunas (documentos) onde o valor é 1 (True/1)
    return resultado_booleano.index[resultado_booleano == 1].tolist()

# Desafio A: (contrato AND rescisao)
if 'contrato' in df_matriz.index and 'rescisao' in df_matriz.index:
    resultado_a = df_matriz.loc['contrato'] & df_matriz.loc['rescisao']
    print(f"Desafio A - (contrato AND rescisao): {buscar_documentos(resultado_a)}")
else:
    print("Termos do Desafio A não encontrados.")

# Desafio B: (multa AND NOT inadimplencia)
if 'multa' in df_matriz.index and 'inadimplencia' in df_matriz.index:
    resultado_b = df_matriz.loc['multa'] & (~df_matriz.loc['inadimplencia'])
    print(f"Desafio B - (multa AND NOT inadimplencia): {buscar_documentos(resultado_b)}")
else:
    print("Termos do Desafio B não encontrados.")

# Desafio C: (sigilo OR propriedade)
if 'sigilo' in df_matriz.index and 'propriedade' in df_matriz.index:
    resultado_c = df_matriz.loc['sigilo'] | df_matriz.loc['propriedade']
    print(f"Desafio C - (sigilo OR propriedade): {buscar_documentos(resultado_c)}")
else:
    print("Termos do Desafio C não encontrados.")

Desafio A - (contrato AND rescisao): ['relatorio_06.txt']
Desafio B - (multa AND NOT inadimplencia): ['relatorio_03.txt', 'relatorio_08.txt']
Desafio C - (sigilo OR propriedade): ['relatorio_07.txt', 'relatorio_08.txt']


## Encerramento
Neste laboratório, implementamos com sucesso o Modelo Booleano clássico. Construímos nossa representação baseada no **Saco de Palavras (Bag of Words)**, geramos o vocabulário, mapeamos as presenças na Matriz de Incidência Binária e demonstramos como a álgebra booleana permite resgatar documentos com base em regras precisas.

Apesar da velocidade e simplicidade desta abordagem, o Modelo Booleano não oferece ordenação (ranqueamento) dos resultados. Todos os documentos retornados possuem a mesma importância. Na próxima etapa da disciplina, avançaremos para o **Modelo Vetorial (Vector Space Model)**, introduzindo conceitos como TF-IDF e Similaridade de Cossenos, o que nos permitirá ordenar os documentos pelo nível de relevância à consulta do usuário.